In [14]:
import pandas as pd
import psycopg2

In [21]:
# Connect to PostgreSQL

DB_CONFIG = {
    "host": "localhost",
    "database": "bank_reviews",
    "user": "postgres",
    "password": "postgres"
}

---
## Step 2: Create the Tables

We connect to `banks_reviews` and define two tables.

### Schema

```sql
banks
  bank_id   SERIAL PRIMARY KEY      -- auto-increments on every insert
  bank_name VARCHAR(255) UNIQUE     -- no duplicate names allowed
  app_nam   VARCHAR(255) UNIQUE     -- no duplicate names allowed

reviews
  review_id       SERIAL PRIMARY KEY
  bank_id   INT REFERENCES banks(bank_id)  -- foreign key
  review_text     TEXT
  rating          INT
  review_date     DATE
  sentiment_label VARCHAR(50)
  sentiment_score VARCHAR(50)
  identified_theme VARCHAR(50)
  source VARCHAR(50)
```

**`CREATE TABLE IF NOT EXISTS`** — safe to re-run; skips silently if the table already exists.  
**`REFERENCES`** — the foreign key means every `banks_id` in `reviews` must exist in `banks`. This is why we create `banks` first.

A **Primary Key (PK)** uniquely identifies each row.  
A **Foreign Key (FK)** links a row in one table to a row in another — here, every review points back to a banks. This enforces **referential integrity**: you cannot add a review for a banks that does not exist.

In [24]:
conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()

# --- banks table ---
# Create banks table
# SERIAL auto-increments the ID on every insert — no need to supply it manually
# UNIQUE ensures no two banks share the same name
cur.execute("""
    CREATE TABLE IF NOT EXISTS banks (
        bank_id   SERIAL PRIMARY KEY,
        bank_name VARCHAR(255) UNIQUE, 
        app_name VARCHAR(255)
    );
""")

# --- reviews table (must come after banks because of the foreign key) ---
# Create reviews table
# bank_id is a FOREIGN KEY — it must match an existing banks.bank_id
# This link is what allows us to JOIN the two tables later

cur.execute("""
    CREATE TABLE IF NOT EXISTS reviews (
        review_id     SERIAL PRIMARY KEY,
        bank_id INT REFERENCES banks(bank_id),
        review_text   TEXT,
        rating        INT,
        review_date   DATE,
        sentiment_label VARCHAR(20),
        sentiment_score FLOAT,
        identified_theme VARCHAR(50),
        source VARCHAR(50)
    );
""")

conn.commit()
cur.close()
conn.close()

print("Tables created successfully.")

Tables created successfully.


---
## Step 3: Load CSV Data into pandas

We read the CSV files into DataFrames **before** touching the database.  
This gives us a chance to inspect and validate the data first.

Think of the DataFrame as a **staging area** — you review the data here before committing it permanently to the database.

In [25]:
df_banks = pd.read_csv("../data/processed/bank_info.csv")
df_reviews     = pd.read_csv("../data/processed/fintech_sentiment_analysis_results.csv")

print(f"Banks: {len(df_banks)} rows")
print(f"Reviews:     {len(df_reviews)} rows")
print()
print(df_banks)
print()
print(df_reviews)

Banks: 3 rows
Reviews:     1390 rows

                     bank_name            app_name
0  Commercial Bank of Ethiopia  CBE Mobile Banking
1                  Dashen Bank    Dashen Super App
2            Bank of Abyssinia  BOA Mobile Banking

         app                                             review  rating  \
0        cbe                                              worst       1   
1        cbe                                 this app very full       5   
2        cbe                                          good apps       4   
3        cbe  this update got crazy i don't know what's goin...       1   
4        cbe                                   thanks for you 😘       5   
...      ...                                                ...     ...   
1385  dashen                          still it's essay for user       4   
1386  dashen  Great app, unfortunately I struggle to use it ...       1   
1387  dashen  It was easy enough before. but now it won't ev...       3   
1388  d

In [ ]:
conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()

# --- Insert banks ---
for _, row in df_banks.iterrows():
    cur.execute(
        """
        INSERT INTO banks (bank_id, bank_name, app_name)
        VALUES (%s, %s, %s)
        ON CONFLICT DO NOTHING;
        """,
        (int(row["bank_id"]), row["bank_name"], row["app_name"])
    )

print(f"Inserted {len(df_banks)} bank rows.")

# --- Insert reviews ---
for _, row in df_reviews.iterrows():
    cur.execute(
        """
        INSERT INTO reviews (review_id, bank_id, review_text, rating, review_date, source)
        VALUES (%s, %s, %s, %s, %s, %s)
        ON CONFLICT DO NOTHING;
        """,
        (
            int(row["review_id"]),
            int(row["bank_id"]),
            row["review_text"],
            int(row["rating"]),
            row["review_date"],
            row["sentiment_label"],
            float(row["sentiment_score"]),
            row["identified_theme"],
            row["source"]
        )
    )

print(f"Inserted {len(df_reviews)} review rows.")

# --- Save and close ---
conn.commit()   # permanently write all inserts to disk
cur.close()
conn.close()

print("Done — connection closed.")

Inserted 3 bank rows.


KeyError: 'review_id'